In [ ]:
import os, random, io
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
from PIL import Image, ImageFilter, ImageEnhance

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision.models import resnet18

from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

DATASET_ROOT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SIZE = 299
RESIZE_SIZE = 320
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

EPOCHS = 10
LR = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

BATCH_PER_SOURCE = 8  # final batch = 8 * (num_real_sources + num_fake_sources)

CAPS = {
    "stylegan_real": 50000,
    "stylegan_fake": 50000,
    "flickr_real": 30000,
    "deepdetect_fake": 30000,
    "wish_real": 1000,
    "wish_fake": 1000,
}

SAVE_PATH = OUT_DIR / "best_resnet18_fft_sourceaware.pth"

Using device: cuda


In [8]:
def rglob_images(p: Path) -> List[Path]:
    if not p.exists():
        return []
    return [x for x in p.rglob("*") if x.is_file() and x.suffix.lower() in IMG_EXTS]

def find_first_dir_contains(substr: str) -> Path:
    substr = substr.lower()
    for p in DATASET_ROOT.rglob("*"):
        if p.is_dir() and substr in str(p).lower():
            return p
    raise FileNotFoundError(f"Could not find folder containing: {substr}")

def cap_list(xs: List[Path], cap: int) -> List[Path]:
    if cap <= 0 or len(xs) <= cap:
        return xs
    return random.sample(xs, cap)

In [14]:
!find /kaggle/input -type d

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/ayushmandatta1
/kaggle/input/datasets/ayushmandatta1/deepdetect-2025
/kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata
/kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata/test
/kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata/test/fake
/kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata/test/real
/kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata/train
/kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata/train/fake
/kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata/train/real
/kaggle/input/datasets/adityajn105
/kaggle/input/datasets/adityajn105/flickr30k
/kaggle/input/datasets/adityajn105/flickr30k/Images
/kaggle/input/datasets/adityajn105/flickr30k/Images/flickr30k_images
/kaggle/input/datasets/wish096
/kaggle/input/datasets/wish096/realvsfake-81k-by-wish
/kaggle/input/datasets/wish096/realvsfake-81k-by-wish/RealVsFake
/kaggle/input/datasets/wish096/realvsfake-81k-b

In [ ]:


# 1) STYLEGAN DATASET
STYLEGAN_ROOT = Path("/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train")

style_real = list((STYLEGAN_ROOT / "real").rglob("*"))
style_fake = list((STYLEGAN_ROOT / "fake").rglob("*"))

style_real = cap_list(style_real, CAPS["stylegan_real"])
style_fake = cap_list(style_fake, CAPS["stylegan_fake"])


# 2) WISH DATASET (realvsfake-81k-by-wish)
WISH_ROOT = Path("/kaggle/input/datasets/wish096/realvsfake-81k-by-wish/RealVsFake/RealVsFake")

wish_real = list((WISH_ROOT / "Real").rglob("*"))
wish_fake = list((WISH_ROOT / "Fake").rglob("*"))

wish_real = cap_list(wish_real, CAPS["wish_real"])
wish_fake = cap_list(wish_fake, CAPS["wish_fake"])


# 3) FLICKR DATASET (REAL ONLY)
FLICKR_ROOT = Path("/kaggle/input/datasets/adityajn105/flickr30k/Images/flickr30k_images")

flickr_real = list(FLICKR_ROOT.rglob("*"))
flickr_real = cap_list(flickr_real, CAPS["flickr_real"])


# 4) DEEPDETECT DATASET (FAKE ONLY FROM TRAIN)
DEEPDETECT_ROOT = Path("/kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata/train")

deepdetect_fake = list((DEEPDETECT_ROOT / "fake").rglob("*"))
deepdetect_fake = cap_list(deepdetect_fake, CAPS["deepdetect_fake"])


def filter_images(paths):
    return [p for p in paths if p.suffix.lower() in IMG_EXTS]

style_real = filter_images(style_real)
style_fake = filter_images(style_fake)
wish_real = filter_images(wish_real)
wish_fake = filter_images(wish_fake)
flickr_real = filter_images(flickr_real)
deepdetect_fake = filter_images(deepdetect_fake)


print("SOURCE COUNTS")
print("stylegan_real    :", len(style_real))
print("stylegan_fake    :", len(style_fake))
print("wish_real        :", len(wish_real))
print("wish_fake        :", len(wish_fake))
print("flickr_real      :", len(flickr_real))
print("deepdetect_fake  :", len(deepdetect_fake))

SOURCE COUNTS
stylegan_real    : 50000
stylegan_fake    : 50000
wish_real        : 1000
wish_fake        : 1000
flickr_real      : 30000
deepdetect_fake  : 30000


In [ ]:
@dataclass(frozen=True)
class Sample:
    path: Path
    label: int     # 0 real, 1 fake
    source: str

def split_source(paths: List[Path], label: int, source: str, train_ratio=0.75, val_ratio=0.10):
    xs = paths[:]
    random.shuffle(xs)
    n = len(xs)
    t = int(train_ratio * n)
    v = int(val_ratio * n)
    return {
        "train": [Sample(p, label, source) for p in xs[:t]],
        "val":   [Sample(p, label, source) for p in xs[t:t+v]],
        "test":  [Sample(p, label, source) for p in xs[t+v:]],
    }

splits: Dict[str, Dict[str, List[Sample]]] = {}

splits["stylegan_real"]   = split_source(style_real, 0, "stylegan_real")
splits["wish_real"]       = split_source(wish_real, 0, "wish_real")
splits["flickr_real"]     = split_source(flickr_real, 0, "flickr_real")

splits["stylegan_fake"]   = split_source(style_fake, 1, "stylegan_fake")
splits["wish_fake"]       = split_source(wish_fake, 1, "wish_fake")
splits["deepdetect_fake"] = split_source(deepdetect_fake, 1, "deepdetect_fake")


train_samples = []
val_samples = []
test_samples = []

for s in splits.values():
    train_samples += s["train"]
    val_samples   += s["val"]
    test_samples  += s["test"]

print("TOTAL SPLITS")
print("Train:", len(train_samples))
print("Val  :", len(val_samples))
print("Test :", len(test_samples))

TOTAL SPLITS
Train: 121500
Val  : 16200
Test : 24300


In [ ]:
def center_crop_resize(img: Image.Image) -> Image.Image:
    img = img.convert("RGB")
    img = img.resize((RESIZE_SIZE, RESIZE_SIZE), Image.BICUBIC)
    left = (RESIZE_SIZE - TARGET_SIZE) // 2
    return img.crop((left, left, left + TARGET_SIZE, left + TARGET_SIZE))

def make_phone_like(img: Image.Image) -> Image.Image:
    if random.random() < 0.7:
        w, h = img.size
        s = random.uniform(0.6, 0.85)
        img = img.resize((int(w*s), int(h*s)), Image.BICUBIC)
        img = img.resize((w, h), Image.BICUBIC)

    if random.random() < 0.4:
        img = img.filter(ImageFilter.MedianFilter(size=3))

    if random.random() < 0.5:
        img = img.filter(ImageFilter.UnsharpMask(radius=2, percent=150))

    if random.random() < 0.3:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.9, 1.2))

    if random.random() < 0.7:
        buf = io.BytesIO()
        q = random.randint(40, 95)
        img.save(buf, format="JPEG", quality=q)
        buf.seek(0)
        img = Image.open(buf).convert("RGB")

    return img

def fft_transform(img_tensor: torch.Tensor) -> torch.Tensor:
    gray = img_tensor.mean(dim=0, keepdim=True)  # (1,H,W)
    fft = torch.fft.fftshift(torch.fft.fft2(gray))
    mag = torch.log1p(torch.abs(fft))
    mag = (mag - mag.min()) / (mag.max() - mag.min() + 1e-8)
    return mag.repeat(3, 1, 1)

def frequency_mask(x: torch.Tensor, p=0.5) -> torch.Tensor:
    if random.random() > p:
        return x
    h, w = x.shape[-2:]
    r = random.randint(10, 40)
    cy, cx = h // 2, w // 2
    y0, y1 = max(0, cy-r), min(h, cy+r)
    x0, x1 = max(0, cx-r), min(w, cx+r)
    x[:, y0:y1, x0:x1] = 0
    return x

In [ ]:
class SourceAwareFFTDataset(Dataset):
    def __init__(self, samples: List[Sample], train: bool):
        self.samples = samples
        self.train = train

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        s = self.samples[idx]
        img = Image.open(s.path).convert("RGB")
        img = center_crop_resize(img)

        if self.train:
            if random.random() < 0.25:
                img = make_phone_like(img)

            if random.random() < 0.15:
                img = img.filter(ImageFilter.GaussianBlur(radius=1.0))

        x = torch.from_numpy(np.array(img)).permute(2,0,1).float() / 255.0
        x = fft_transform(x)

        if self.train:
            x = frequency_mask(x, p=0.5)

        return x, torch.tensor(s.label, dtype=torch.long), s.source

In [ ]:
class BalancedSourceBatchSampler(Sampler[List[int]]):
    def __init__(self, samples: List[Sample], batch_per_source: int):
        self.samples = samples
        self.batch_per_source = batch_per_source

        self.pools: Dict[Tuple[int, str], List[int]] = {}
        for i, s in enumerate(samples):
            self.pools.setdefault((s.label, s.source), []).append(i)

        self.real_sources = sorted({s.source for s in samples if s.label == 0})
        self.fake_sources = sorted({s.source for s in samples if s.label == 1})

        if not self.real_sources or not self.fake_sources:
            raise ValueError("Need at least 1 real source and 1 fake source.")

        def batches_for(label: int, srcs: List[str]) -> int:
            return min(len(self.pools[(label, s)]) for s in srcs) // self.batch_per_source

        self.num_batches = min(
            batches_for(0, self.real_sources),
            batches_for(1, self.fake_sources),
        )

    def __len__(self):
        return self.num_batches

    def __iter__(self):
        pools = {k: v[:] for k, v in self.pools.items()}
        for k in pools:
            random.shuffle(pools[k])

        for b in range(self.num_batches):
            batch = []
            for s in self.real_sources:
                start = b * self.batch_per_source
                batch.extend(pools[(0, s)][start:start + self.batch_per_source])

            for s in self.fake_sources:
                start = b * self.batch_per_source
                batch.extend(pools[(1, s)][start:start + self.batch_per_source])

            random.shuffle(batch)
            yield batch

In [30]:
from torch.utils.data import WeightedRandomSampler

labels = [s.label for s in train_samples]
class_counts = np.bincount(labels)
class_weights = 1.0 / class_counts
sample_weights = [class_weights[label] for label in labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_samples),
    replacement=True
)

train_loader = DataLoader(
    train_ds,
    batch_size=64,
    sampler=sampler,      # <-- CORRECT
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=64,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_ds,
    batch_size=64,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print("Steps per epoch:", len(train_loader))

Steps per epoch: 1899


In [21]:
model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

In [ ]:
@torch.no_grad()
def eval_loader(loader):
    model.eval()
    y_true, y_pred = [], []
    per_source = {}

    for x, y, src in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        out = model(x)
        p = out.argmax(1)

        yt = y.cpu().numpy().tolist()
        yp = p.cpu().numpy().tolist()
        src = list(src)

        y_true.extend(yt)
        y_pred.extend(yp)

        for s, t_i, p_i in zip(src, yt, yp):
            per_source.setdefault(s, {"t": [], "p": []})
            per_source[s]["t"].append(t_i)
            per_source[s]["p"].append(p_i)

    def metrics(t, p):
        acc = accuracy_score(t, p)
        pr, rc, f1, _ = precision_recall_fscore_support(t, p, average="binary", zero_division=0)
        return {"acc": float(acc), "precision": float(pr), "recall": float(rc), "f1": float(f1)}

    overall = metrics(y_true, y_pred)
    per_src_metrics = {s: metrics(d["t"], d["p"]) for s, d in per_source.items()}

    worst_f1 = min(m["f1"] for m in per_src_metrics.values()) if per_src_metrics else overall["f1"]

    return overall, per_src_metrics, worst_f1


def print_metrics(title, overall, per_src):
    print(f"\n=== {title} ===")
    print("OVERALL:", {k: round(v, 4) for k, v in overall.items()})
    for s in sorted(per_src.keys()):
        m = per_src[s]
        print(f"  - {s:15s} acc={m['acc']:.4f} f1={m['f1']:.4f} p={m['precision']:.4f} r={m['recall']:.4f}")

In [ ]:
best_score = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for x, y, _ in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item())

    print(f"\nEpoch {epoch}: train_loss={total_loss / max(1, len(train_loader)):.4f}")

    val_overall, val_per_src, val_worst_f1 = eval_loader(val_loader)
    print_metrics("VAL", val_overall, val_per_src)
    print("VAL worst-source F1:", round(val_worst_f1, 4))

    if val_worst_f1 > best_score:
        best_score = val_worst_f1
        best_state = {
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "best_score": best_score,
            "config": {
                "EPOCHS": EPOCHS,
                "LR": LR,
                "WEIGHT_DECAY": WEIGHT_DECAY,
                "TARGET_SIZE": TARGET_SIZE,
                "RESIZE_SIZE": RESIZE_SIZE,
                "BATCH_PER_SOURCE": BATCH_PER_SOURCE,
                "CAPS": CAPS,
                "SEED": SEED
            }
        }
        torch.save(best_state, SAVE_PATH)
        print(f"✅ Saved NEW BEST model to: {SAVE_PATH} (score={best_score:.4f})")

print("\nTraining done.")
print("Best saved checkpoint:", SAVE_PATH)

Epoch 1/10: 100%|██████████| 1899/1899 [17:26<00:00,  1.81it/s]


Epoch 1: train_loss=0.3769



=== VAL ===
OVERALL: {'acc': 0.7635, 'precision': 0.6875, 'recall': 0.9664, 'f1': 0.8034}
  - deepdetect_fake acc=0.9653 f1=0.9824 p=1.0000 r=0.9653
  - flickr_real     acc=0.9387 f1=0.0000 p=0.0000 r=0.0000
  - stylegan_fake   acc=0.9682 f1=0.9838 p=1.0000 r=0.9682
  - stylegan_real   acc=0.3372 f1=0.0000 p=0.0000 r=0.0000
  - wish_fake       acc=0.9100 f1=0.9529 p=1.0000 r=0.9100
  - wish_real       acc=0.3900 f1=0.0000 p=0.0000 r=0.0000
VAL worst-source F1: 0.0
✅ Saved NEW BEST model to: /kaggle/working/best_resnet18_fft_sourceaware.pth (score=0.0000)


Epoch 2/10: 100%|██████████| 1899/1899 [17:39<00:00,  1.79it/s]


Epoch 2: train_loss=0.3520


...


Epoch 9: train_loss=0.1947



=== VAL ===
OVERALL: {'acc': 0.8196, 'precision': 0.8627, 'recall': 0.7602, 'f1': 0.8082}
  - deepdetect_fake acc=0.7840 f1=0.8789 p=1.0000 r=0.7840
  - flickr_real     acc=0.9910 f1=0.0000 p=0.0000 r=0.0000
  - stylegan_fake   acc=0.7470 f1=0.8552 p=1.0000 r=0.7470
  - stylegan_real   acc=0.8136 f1=0.0000 p=0.0000 r=0.0000
  - wish_fake       acc=0.7100 f1=0.8304 p=1.0000 r=0.7100
  - wish_real       acc=0.7900 f1=0.0000 p=0.0000 r=0.0000
VAL worst-source F1: 0.0


Epoch 10/10: 100%|██████████| 1899/1899 [17:57<00:00,  1.76it/s]


Epoch 10: train_loss=0.1832



=== VAL ===
OVERALL: {'acc': 0.8005, 'precision': 0.8927, 'recall': 0.6831, 'f1': 0.774}
  - deepdetect_fake acc=0.7093 f1=0.8300 p=1.0000 r=0.7093
  - flickr_real     acc=0.9997 f1=0.0000 p=0.0000 r=0.0000
  - stylegan_fake   acc=0.6694 f1=0.8020 p=1.0000 r=0.6694
  - stylegan_real   acc=0.8686 f1=0.0000 p=0.0000 r=0.0000
  - wish_fake       acc=0.5800 f1=0.7342 p=1.0000 r=0.5800
  - wish_real       acc=0.9300 f1=0.0000 p=0.0000 r=0.0000
VAL worst-source F1: 0.0

Training done.
Best saved checkpoint: /kaggle/working/best_resnet18_fft_sourceaware.pth


In [35]:
epoch = 10

In [36]:
LATEST_PATH = OUT_DIR / "latest_resnet18_fft_sourceaware.pth"

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epoch": epoch,  # if saving inside loop, this will be current epoch
    "config": {
        "EPOCHS": EPOCHS,
        "LR": LR,
        "WEIGHT_DECAY": WEIGHT_DECAY,
        "TARGET_SIZE": TARGET_SIZE,
        "RESIZE_SIZE": RESIZE_SIZE,
        "CAPS": CAPS,
        "SEED": SEED
    }
}, LATEST_PATH)

print("✅ Saved latest checkpoint to:", LATEST_PATH)

✅ Saved latest checkpoint to: /kaggle/working/latest_resnet18_fft_sourceaware.pth


In [37]:
EPOCHS_EXTRA = 5

In [ ]:
ckpt = torch.load(LATEST_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
optimizer.load_state_dict(ckpt["optimizer_state_dict"])

start_epoch = ckpt.get("epoch", 10) + 1
end_epoch = start_epoch + EPOCHS_EXTRA - 1

print(f"Resuming from epoch {start_epoch-1} -> training until epoch {end_epoch}")

for epoch in range(start_epoch, end_epoch + 1):
    model.train()
    total_loss = 0.0

    for x, y, _ in tqdm(train_loader, desc=f"Epoch {epoch}"):
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item())

    print(f"\nEpoch {epoch}: train_loss={total_loss / max(1, len(train_loader)):.4f}")

    val_overall, val_per_src, val_worst_f1 = eval_loader(val_loader)
    print_metrics("VAL", val_overall, val_per_src)
    print("VAL worst-source F1:", round(val_worst_f1, 4))

    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch,
    }, LATEST_PATH)

    print("✅ Saved latest checkpoint:", LATEST_PATH)

Resuming from epoch 10 -> training until epoch 15


Epoch 11: 100%|██████████| 1899/1899 [17:25<00:00,  1.82it/s]


Epoch 11: train_loss=0.1757



=== VAL ===
OVERALL: {'acc': 0.8626, 'precision': 0.842, 'recall': 0.8927, 'f1': 0.8666}
  - deepdetect_fake acc=0.9163 f1=0.9563 p=1.0000 r=0.9163
  - flickr_real     acc=0.9900 f1=0.0000 p=0.0000 r=0.0000
  - stylegan_fake   acc=0.8808 f1=0.9366 p=1.0000 r=0.8808
  - stylegan_real   acc=0.7400 f1=0.0000 p=0.0000 r=0.0000
  - wish_fake       acc=0.7800 f1=0.8764 p=1.0000 r=0.7800
  - wish_real       acc=0.7300 f1=0.0000 p=0.0000 r=0.0000
VAL worst-source F1: 0.0
✅ Saved latest checkpoint: /kaggle/working/latest_resnet18_fft_sourceaware.pth


Epoch 12: 100%|██████████| 1899/1899 [17:57<00:00,  1.76it/s]

...


Epoch 14: train_loss=0.1511



=== VAL ===
OVERALL: {'acc': 0.8298, 'precision': 0.8219, 'recall': 0.842, 'f1': 0.8318}
  - deepdetect_fake acc=0.8593 f1=0.9243 p=1.0000 r=0.8593
  - flickr_real     acc=0.9553 f1=0.0000 p=0.0000 r=0.0000
  - stylegan_fake   acc=0.8314 f1=0.9079 p=1.0000 r=0.8314
  - stylegan_real   acc=0.7374 f1=0.0000 p=0.0000 r=0.0000
  - wish_fake       acc=0.8500 f1=0.9189 p=1.0000 r=0.8500
  - wish_real       acc=0.6900 f1=0.0000 p=0.0000 r=0.0000
VAL worst-source F1: 0.0
✅ Saved latest checkpoint: /kaggle/working/latest_resnet18_fft_sourceaware.pth


Epoch 15: 100%|██████████| 1899/1899 [17:57<00:00,  1.76it/s]


Epoch 15: train_loss=0.1446



=== VAL ===
OVERALL: {'acc': 0.784, 'precision': 0.9219, 'recall': 0.6205, 'f1': 0.7417}
  - deepdetect_fake acc=0.6530 f1=0.7901 p=1.0000 r=0.6530
  - flickr_real     acc=0.9980 f1=0.0000 p=0.0000 r=0.0000
  - stylegan_fake   acc=0.6022 f1=0.7517 p=1.0000 r=0.6022
  - stylegan_real   acc=0.9174 f1=0.0000 p=0.0000 r=0.0000
  - wish_fake       acc=0.5600 f1=0.7179 p=1.0000 r=0.5600
  - wish_real       acc=0.9300 f1=0.0000 p=0.0000 r=0.0000
VAL worst-source F1: 0.0
✅ Saved latest checkpoint: /kaggle/working/latest_resnet18_fft_sourceaware.pth


In [43]:
!ls /kaggle/input/datasets/kashirhanif/frequency-model-checkpoint

fft_binary_resnet18.pth		       freq_v5_epoch2_checkpoint.pt
fft_debiased_resnet18_fromscratch.pth  latest_resnet18_fft_sourceaware.pth


In [ ]:
CKPT_PATH = "/kaggle/input/datasets/kashirhanif/frequency-model-checkpoint/latest_resnet18_fft_sourceaware.pth"

ckpt = torch.load(CKPT_PATH, map_location=DEVICE)

if "model_state_dict" in ckpt:
    model.load_state_dict(ckpt["model_state_dict"])
else:
    model.load_state_dict(ckpt)

model.to(DEVICE)
model.eval()

print("✅ Loaded checkpoint from:", CKPT_PATH)

✅ Loaded checkpoint from: /kaggle/input/datasets/kashirhanif/frequency-model-checkpoint/latest_resnet18_fft_sourceaware.pth


In [45]:
test_overall, test_per_src, test_worst_f1 = eval_loader(test_loader)

print_metrics("TEST (EPOCH 11 MODEL)", test_overall, test_per_src)
print("TEST worst-source F1:", round(test_worst_f1, 4))


=== TEST (EPOCH 11 MODEL) ===
OVERALL: {'acc': 0.8651, 'precision': 0.8475, 'recall': 0.8905, 'f1': 0.8684}
  - deepdetect_fake acc=0.9182 f1=0.9574 p=1.0000 r=0.9182
  - flickr_real     acc=0.9887 f1=0.0000 p=0.0000 r=0.0000
  - stylegan_fake   acc=0.8752 f1=0.9334 p=1.0000 r=0.8752
  - stylegan_real   acc=0.7503 f1=0.0000 p=0.0000 r=0.0000
  - wish_fake       acc=0.8200 f1=0.9011 p=1.0000 r=0.8200
  - wish_real       acc=0.8467 f1=0.0000 p=0.0000 r=0.0000
TEST worst-source F1: 0.0
